In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer

spark = SparkSession.builder \
    .appName("TugasPertemuan11") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [3]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [11]:
# soal no 1
df = spark.createDataFrame(data_tugas11)

indexer_nama = StringIndexer(inputCol="nama_menu", outputCol="nama_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

df_indexed = indexer_nama.fit(df).transform(df)
df_indexed = indexer_cabang.fit(df).transform(df_indexed)
df_indexed = indexer_waktu.fit(df).transform(df_indexed)

df_indexed.select("nama_menu", "nama_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_idx").show(6)


+--------------+--------+----------+----------+-----------+---------+
|     nama_menu|nama_idx|    cabang|cabang_idx|waktu_pesan|waktu_idx|
+--------------+--------+----------+----------+-----------+---------+
|Kentang Goreng|     5.0|  Semarang|       2.0|      Malam|      1.0|
|Kentang Goreng|     5.0|Yogyakarta|       1.0|       Sore|      3.0|
|Kentang Goreng|     5.0|  Semarang|       2.0|      Siang|      2.0|
|Kentang Goreng|     5.0|Yogyakarta|       1.0|       Pagi|      0.0|
|    Roti Bakar|     4.0|  Magelang|       0.0|      Malam|      1.0|
|     Americano|     0.0|  Magelang|       0.0|       Pagi|      0.0|
+--------------+--------+----------+----------+-----------+---------+
only showing top 6 rows



In [12]:
# soal no 1
assembler_tugas = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_final = assembler_tugas.transform(df_indexed)

df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [15]:
# soal no 2
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1825
Jumlah data testing   : 675
Model berhasil dilatih!
Koefisien tiap fitur: [21899.699695329688,3.0070528126196643,-85.74425311496488,206.6234541825447,281.8759328790849]
Intercept: -65835.73823558443


In [24]:
# soal no 3

predictions = model.transform(test)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

print("\nInterpretasi Model: \nModel regresi linear ini cukup baik dan layak digunakan untuk memprediksi omzet karena nilai R² tergolong tinggi (mendekati 1), yang menunjukkan bahwa sebagian besar variasi total penjualan berhasil dijelaskan oleh fitur-fitur yang ada. Selain itu, nilai RMSE yang dihasilkan relatif kecil dibandingkan rata-rata nominal penjualan per transaksi sehingga margin kesalahannya masih dalam batas wajar.")

RMSE : Rp 10,670
R²   : 0.9131  (91.3% variasi data berhasil dijelaskan model)

Interpretasi Model: 
Model regresi linear ini cukup baik dan layak digunakan untuk memprediksi omzet karena nilai R² tergolong tinggi (mendekati 1), yang menunjukkan bahwa sebagian besar variasi total penjualan berhasil dijelaskan oleh fitur-fitur yang ada. Selain itu, nilai RMSE yang dihasilkan relatif kecil dibandingkan rata-rata nominal penjualan per transaksi sehingga margin kesalahannya masih dalam batas wajar.


In [28]:
# soal no 4

# 1. Menggabungkan hanya 2 fitur numerik (tanpa nama_menu, cabang, waktu_pesan)
assembler_dua_fitur = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)
df_dua_fitur = assembler_dua_fitur.transform(df)

# 2. Split data dengan proporsi dan seed yang sama dengan Bagian B (75:25, seed=101)
train_2, test_2 = df_dua_fitur.randomSplit([0.75, 0.25], seed=101)

# 3. Melatih model LinearRegression kedua
lr_2 = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_2 = lr_2.fit(train_2)

# 4. Evaluasi R² (dan RMSE) pada data testing
predictions_2 = model_2.transform(test_2)

evaluator_r2_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_2 = evaluator_r2_2.evaluate(predictions_2)

evaluator_rmse_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse_2 = evaluator_rmse_2.evaluate(predictions_2)

# 5. Perbandingan R² Model Utama (Bagian C) vs Model Kedua (Bagian D)
selisih_r2 = r2 - r2_2
print("=== PERBANDINGAN NILAI R² ===")
print(f"R² Model Utama (5 Fitur) : {r2:.4f} ({r2*100:.2f}%)")
print(f"R² Model Kedua (2 Fitur) : {r2_2:.4f} ({r2_2*100:.2f}%)")
print(f"Selisih R²               : {selisih_r2:.4f} ({selisih_r2*100:.2f}%)")

print("\n1. Perbandingan Nilai R²:Nilai R² pada model kedua yang hanya menggunakan 2 fitur (`qty` dan `harga_satuan`) hampir identik dengan nilai R² model utama pada Bagian C (selisihnya sangat kecil, mendekati 0). ")
print("\n2. Pengaruh Penambahan Fitur Kategorikal: Menambahkan informasi `nama_menu`, `cabang`, dan `waktu_pesan` tidak meningkatkan akurasi prediksi secara berarti. Hal ini terjadi karena dalam dataset transaksi ini, besaran `total_penjualan` secara langsung ditentukan oleh perkalian `qty` dan `harga_satuan` (ditambah variasi acak kecil), sedangkan perbedaan menu, cabang, maupun waktu pemesanan tidak memengaruhi perhitungan nominal penjualan.")

=== PERBANDINGAN NILAI R² ===
R² Model Utama (5 Fitur) : 0.9131 (91.31%)
R² Model Kedua (2 Fitur) : 0.9129 (91.29%)
Selisih R²               : 0.0002 (0.02%)

1. Perbandingan Nilai R²:Nilai R² pada model kedua yang hanya menggunakan 2 fitur (`qty` dan `harga_satuan`) hampir identik dengan nilai R² model utama pada Bagian C (selisihnya sangat kecil, mendekati 0). 

2. Pengaruh Penambahan Fitur Kategorikal: Menambahkan informasi `nama_menu`, `cabang`, dan `waktu_pesan` tidak meningkatkan akurasi prediksi secara berarti. Hal ini terjadi karena dalam dataset transaksi ini, besaran `total_penjualan` secara langsung ditentukan oleh perkalian `qty` dan `harga_satuan` (ditambah variasi acak kecil), sedangkan perbedaan menu, cabang, maupun waktu pemesanan tidak memengaruhi perhitungan nominal penjualan.


In [29]:
# soal no 4

# 1. Membuat DataFrame baru berisi 3 skenario pesanan sesuai kategori & kolom dataset awal
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 20000),
    ("Roti Bakar", "Yogyakarta", "Sore", 4, 25000),
    ("Es Teh", "Semarang", "Siang", 1, 15000)
]
kolom_skenario = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(data_skenario, kolom_skenario)

# 2. Terapkan StringIndexer yang sama dari Bagian A (di-fit pada data awal df, BUKAN df_skenario)
skenario_indexed = indexer_nama.fit(df).transform(df_skenario)
skenario_indexed = indexer_cabang.fit(df).transform(skenario_indexed)
skenario_indexed = indexer_waktu.fit(df).transform(skenario_indexed)

# 3. Gabungkan fitur menggunakan VectorAssembler yang sama dari Bagian A
# (Sesuaikan nama variabel assembler_tugas jika di Bagian A kamu menamainya berbeda)
skenario_final = assembler_tugas.transform(skenario_indexed)

# 4. Prediksi total_penjualan menggunakan model utama (5 fitur) dari Bagian B
prediksi_skenario = model.transform(skenario_final)

# 5. Tampilkan hasil prediksi ketiga skenario
prediksi_skenario.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |38018.973154353254|
|Roti Bakar|Yogyakarta|Sore       |4  |25000       |97648.65510158587 |
|Es Teh    |Semarang  |Siang      |1  |15000       |1889.519663818588 |
+----------+----------+-----------+---+------------+------------------+

